In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import os

# Try to set HF mirror as fallback
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

try:
    # Step 1: Load data
    train = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
    test = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')
    submission = pd.read_csv('/kaggle/input/llm-classification-finetuning/sample_submission.csv')

    # Step 2: Try loading model with various fallbacks
    try:
        model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
    except:
        print("Primary download failed, trying mirror...")
        os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
        model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

    # Step 3: Generate embeddings
    def get_embedding(row):
        text_a = str(row['response_a'])
        text_b = str(row['response_b'])
        emb_a = model.encode(text_a)
        emb_b = model.encode(text_b)
        return np.concatenate([emb_a, emb_b])

    X = np.vstack(train.apply(get_embedding, axis=1))
    y = np.argmax(train[['winner_model_a', 'winner_model_b', 'winner_tie']].values, axis=1)

    # Step 4: Train model
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)

    # Step 5: Evaluate
    y_pred_val = clf.predict_proba(X_val)
    val_loss = log_loss(y_val, y_pred_val)
    print(f"Validation Log Loss: {val_loss:.4f}")

    # Step 6: Predict on test
    X_test = np.vstack(test.apply(get_embedding, axis=1))
    test_preds = clf.predict_proba(X_test)
    submission[['winner_model_a', 'winner_model_b', 'winner_tie']] = test_preds
    submission_path = '/kaggle/working/submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f"✅ submission.csv saved at: {submission_path}")
    print("Files in working directory:", os.listdir('/kaggle/working'))


except Exception as e:
    print(f"Error: {e}")
    print("\nIf the error is related to model downloading, try:")
    print("1. Checking your internet connection")
    print("2. Using a VPN if you're in a restricted network")
    print("3. Downloading the model files separately on a machine with internet")
    print("4. Using local_files_only=True if you have cached model files")